In [2]:
import pandas as pd
import numpy as np

In [3]:
# load transaction data
df_transaction = pd.read_csv('transaction_data.csv')
# check number of rows in the transaction data
rows_transaction = len(df_transaction)
print(f'The number of rows in the transaction data is: {rows_transaction}')

# load product data
df_product = pd.read_csv('product.csv')
rows_product = len(df_product)
print(f'The number of rows in the product data is: {rows_product}')

The number of rows in the transaction data is: 2595732
The number of rows in the product data is: 92353


In [4]:
# check for missing values in the transaction data
df_transaction.isna().sum()

# you're gonna get the number of missing values in each column of the transaction data and print it out. You can use the isna() method to check for missing values and the sum() method to count them.

##

household_key        0
BASKET_ID            0
DAY                  0
PRODUCT_ID           0
QUANTITY             0
SALES_VALUE          0
STORE_ID             0
RETAIL_DISC          0
TRANS_TIME           0
WEEK_NO              0
COUPON_DISC          0
COUPON_MATCH_DISC    0
dtype: int64

In [5]:
# check for missing values in the product data
df_product.isna().sum()


PRODUCT_ID              0
MANUFACTURER            0
DEPARTMENT              0
BRAND                   0
COMMODITY_DESC          0
SUB_COMMODITY_DESC      0
CURR_SIZE_OF_PRODUCT    0
dtype: int64

In [6]:
df_product["DEPARTMENT"].value_counts()

DEPARTMENT
GROCERY            39021
DRUG GM            31529
PRODUCE             3118
COSMETICS           3011
NUTRITION           2914
MEAT                2544
MEAT-PCKGD          2427
DELI                2354
PASTRY              2149
FLORAL               938
SEAFOOD-PCKGD        563
MISC. TRANS.         490
SPIRITS              377
SEAFOOD              369
GARDEN CENTER        128
RESTAURANT           102
MISC SALES TRAN       88
SALAD BAR             48
COUP/STR & MFG        39
TRAVEL & LEISUR       28
FROZEN GROCERY        23
KIOSK-GAS             16
                      15
CHEF SHOPPE           14
RX                     9
CNTRL/STORE SUP        4
GM MERCH EXP           3
TOYS                   3
DAIRY DELI             3
VIDEO RENTAL           3
POSTAL CENTER          3
GRO BAKERY             2
CHARITABLE CONT        2
PROD-WHS SALES         2
DELI/SNACK BAR         2
AUTOMOTIVE             2
PHOTO                  2
VIDEO                  2
ELECT &PLUMBING        1
MEAT-WHSE     

In [7]:
# merge the two dataframes on the PRODUCT_ID column
df = df_transaction.merge(df_product, on="PRODUCT_ID", how="left")
print(len(df_transaction), len(df))   # these should match, safety check making sure we didn't lose any rows in the merge
df.groupby("DEPARTMENT")["SALES_VALUE"].sum().sort_values(ascending=False).head(15)

2595732 2595732


DEPARTMENT
GROCERY            4093814.14
DRUG GM            1055358.03
PRODUCE             557452.11
MEAT                548786.81
KIOSK-GAS           544222.28
MEAT-PCKGD          412436.77
DELI                260866.51
PASTRY              121739.86
MISC SALES TRAN     119960.04
NUTRITION            97669.04
SEAFOOD-PCKGD        63093.62
FLORAL               39653.34
COSMETICS            32360.37
SALAD BAR            30953.30
SEAFOOD              29496.92
Name: SALES_VALUE, dtype: float64

In [8]:
# checking the top 10 commodities in the MISC SALES TRAN department
df[df["DEPARTMENT"] == "MISC SALES TRAN"]["COMMODITY_DESC"].value_counts().head(10)

COMMODITY_DESC
COUPON/MISC ITEMS    6050
Name: count, dtype: int64

In [9]:
# exclude the KIOSK-GAS and MISC SALES TRAN departments from the analysis
before = len(df)
df_clean = df[~df["DEPARTMENT"].isin(["KIOSK-GAS", "MISC SALES TRAN"])]
print(before, len(df_clean), before - len(df_clean))

2595732 2567623 28109


## Cleaning Log

### 1. Excluded KIOSK-GAS department
- Rows removed: 22,059 transaction rows (~0.8% of rows)
- Sales removed: $544,222 (~6.8% of total sales)
- Reason: Gas is not a grocery product and doesn't respond to
  promotions the way groceries do. Including it would blur the
  discount and promotion results.
- Note: It's only 16 products, but it's a large share of sales because
  each gas purchase is a big dollar amount. That's why it was excluded
  despite the small row count.

### 2. Excluded MISC SALES TRAN department
- Rows removed: 6,050 transaction rows
- Content: all rows have COMMODITY_DESC = "COUPON/MISC ITEMS"
- Sales removed: $119,960 (~1.5% of total sales)
- Reason: These are coupon and miscellaneous bookkeeping entries, not
  products customers chose to buy, so they don't belong in a product
  or discount analysis.
- Caveat: Coupons are part of this project. They'll be analyzed
  separately using coupon_redempt and the COUPON_DISC columns.

### Summary after steps 1-2
- Rows before: 2,595,732 | Rows after: 2,567,623
- Total removed: 28,109 rows (~1.1% of the data)
- Merge check: transaction and merged row counts matched (no duplicates)

In [10]:
# check how many rows were excluded from the KIOSK-GAS department
(df["DEPARTMENT"] == "KIOSK-GAS").sum()

np.int64(22059)

In [ ]:
# here we are checking for rows where the QUANTITY or SALES_VALUE is zero, and then printing out the counts of those rows. We also check how many rows have both QUANTITY and SALES_VALUE equal to zero.
zero_qty = df_clean["QUANTITY"] == 0
zero_sales = df_clean["SALES_VALUE"] == 0
print(zero_qty.sum(), zero_sales.sum(), (zero_qty & zero_sales).sum())

print(df_clean["QUANTITY"].quantile([.5, .99, .999, .9999]))
print((df_clean["QUANTITY"] > 50).sum())

14409 18786 14342
0.5000     1.0
0.9900     5.0
0.9990    10.0
0.9999    23.0
Name: QUANTITY, dtype: float64
16


In [ ]:
# here we are checking for rows where the SALES_VALUE is zero and QUANTITY is positive
# basically someone bought something but the sales value was zero, WEIRD !
# printing out the first 5 rows of those cases, and only showing the columns that are relevant to this analysis
cols = ["QUANTITY", "SALES_VALUE", "RETAIL_DISC", "DEPARTMENT", "COMMODITY_DESC"]
df_clean[(df_clean["SALES_VALUE"] == 0) & (df_clean["QUANTITY"] > 0)][cols].head(5)

,QUANTITY,SALES_VALUE,RETAIL_DISC,DEPARTMENT,COMMODITY_DESC
766,1,0.0,0.00,GROCERY,BAG SNACKS
996,1,0.0,-1.48,PRODUCE,VEGETABLES - ALL OTHERS
1095,1,0.0,-3.29,GROCERY,SPICES & EXTRACTS
1113,1,0.0,-2.49,GROCERY,SPICES & EXTRACTS
1161,1,0.0,-1.19,GROCERY,VEGETABLES - SHELF STABLE


In [ ]:
# printing out the number of rows where SALES_VALUE is zero and QUANTITY is positive, and also printing out the proportion of those rows where RETAIL_DISC is negative
free = df_clean[(df_clean["SALES_VALUE"] == 0) & (df_clean["QUANTITY"] > 0)]
# printing out the number of rows in the free dataframe, and also printing out the proportion of those rows where RETAIL_DISC is negative
print(len(free), (free["RETAIL_DISC"] < 0).mean())

4444 0.8433843384338434


### 3. Kept: zero-sales rows with quantity > 0 (fully discounted items)
- Rows: 4,444 (~0.17% of 2,567,623 rows)
- Pattern: QUANTITY ≥ 1 but SALES_VALUE = 0. About [84]% have a
  negative RETAIL_DISC equal to the item's price, meaning the store
  discounted the item 100% (e.g., a free item in a promotion).
- Decision: Kept. These are real purchase events and 100% discounts,
  which are directly relevant to the question of whether discounts
  generate sales.
- Caveat: SALES_VALUE = 0 breaks calculations that divide by the
  amount paid. Pre-discount price will be reconstructed as
  (SALES_VALUE - RETAIL_DISC - COUPON_MATCH_DISC) / QUANTITY instead.
- Open item: The remaining [16]% have RETAIL_DISC = 0. Investigating
  [what they are, based on department and coupon checks]. Decision: [pending].

### 4. Zero-quantity rows: [pending]
- Rows: 14,409 with QUANTITY = 0; 14,342 of those also have SALES_VALUE = 0

In [ ]:
# checking for rows where RETAIL_DISC is zero (no discount)
no_disc = free[free["RETAIL_DISC"] == 0]
print(len(no_disc))
print(no_disc["DEPARTMENT"].value_counts().head(8))

696
DEPARTMENT
COUP/STR & MFG    444
GROCERY           232
DRUG GM             5
PASTRY              5
PRODUCE             4
GARDEN CENTER       1
MEAT                1
FLORAL              1
Name: count, dtype: int64


In [ ]:
# checking for rows where COUPON_DISC or COUPON_MATCH_DISC is negative in the no_disc dataframe, and printing out the proportion of those rows
print((no_disc["COUPON_DISC"] < 0).mean(), (no_disc["COUPON_MATCH_DISC"] < 0).mean())

0.0014367816091954023 0.0014367816091954023


In [ ]:
# checking for rows where the DEPARTMENT is COUP/STR & MFG, and printing out the top 10 commodities in that department, as well as the number of rows and the total sales value for that department
cc = df_clean[df_clean["DEPARTMENT"] == "COUP/STR & MFG"]
print(cc["COMMODITY_DESC"].value_counts().head(10))
print(len(cc), cc["SALES_VALUE"].sum())

COMMODITY_DESC
COUPONS/STORE & MFG    817
Name: count, dtype: int64
817 1027.7199999999998


In [ ]:
# removing rows where the DEPARTMENT is COUP/STR & MFG
before = len(df_clean)
df_clean = df_clean[df_clean["DEPARTMENT"] != "COUP/STR & MFG"]
print(before, len(df_clean), before - len(df_clean))

2567623 2566806 817


### 4. Excluded COUP/STR & MFG department
- Rows removed: 817 (~$1,028 in sales)
- Content: all rows have COMMODITY_DESC = "COUPONS/STORE & MFG"
- Reason: Coupon bookkeeping entries, not products customers bought.
  Same reasoning as MISC SALES TRAN. Coupon information is kept in
  the discount columns and in coupon_redempt.
- Caveat: Interpretation inferred from the data, not confirmed in
  the dataset documentation.

In [ ]:
# checking for rows where SALES_VALUE is zero and QUANTITY is positive, and also checking how many of those rows have RETAIL_DISC equal to zero
free = df_clean[(df_clean["SALES_VALUE"] == 0) & (df_clean["QUANTITY"] > 0)]
print(len(free), (free["RETAIL_DISC"] == 0).sum())

4000 252


In [ ]:
# creating a mask for rows where SALES_VALUE is zero and QUANTITY is positive, and all discount columns are zero
mask = ((df_clean["SALES_VALUE"] == 0) & (df_clean["QUANTITY"] > 0) &
        (df_clean["RETAIL_DISC"] == 0) & (df_clean["COUPON_DISC"] == 0) &
        (df_clean["COUPON_MATCH_DISC"] == 0))

print(mask.sum())   # about 252

251


In [ ]:
# removing those rows from the dataframe
before = len(df_clean)
df_clean = df_clean[~mask]
print(before, len(df_clean), before - len(df_clean))

2566806 2566555 251


### 5. Dropped: unexplained zero-price items (no discount, no coupon)
- Rows removed: 251 (~0.01% of rows)
- Pattern: QUANTITY ≥ 1 and SALES_VALUE = 0, but RETAIL_DISC,
  COUPON_DISC and COUPON_MATCH_DISC are all 0.
- Reason: Unlike the fully discounted items (kept), nothing in the
  data explains why these were free. They can't be treated as
  discount events and their price can't be reconstructed.
- Caveat: Could be giveaways or recording errors; the cause is
  unknown. Share is negligible, so impact on results should be minimal.

In [ ]:
# checking for rows where QUANTITY is zero, and printing out the number of those rows, as well as the descriptive statistics for the SALES_VALUE column in those rows
zero_qty = df_clean["QUANTITY"] == 0
print(zero_qty.sum())
print(df_clean[zero_qty]["SALES_VALUE"].describe())

14406
count    14406.000000
mean         0.001338
std          0.060092
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          5.820000
Name: SALES_VALUE, dtype: float64


In [ ]:
# checking for rows where QUANTITY is zero and SALES_VALUE is positive, and printing out the number of those rows, as well as the descriptive statistics for the SALES_VALUE column in those rows
odd = df_clean[(df_clean["QUANTITY"] == 0) & (df_clean["SALES_VALUE"] > 0)]
print(len(odd))
odd[["QUANTITY", "SALES_VALUE", "RETAIL_DISC", "COUPON_DISC", "DEPARTMENT"]].head(10)

67


,QUANTITY,SALES_VALUE,RETAIL_DISC,COUPON_DISC,DEPARTMENT
6032,0,7.000000e-02,0.0,0.0,DELI
136410,0,4.000000e-02,0.0,0.0,PRODUCE
211149,0,5.551115e-17,0.0,0.0,GROCERY
211156,0,5.551115e-17,0.0,0.0,GROCERY
304499,0,6.661338e-16,0.0,0.0,GROCERY
418897,0,2.220446e-16,0.0,0.0,GROCERY
479956,0,4.440892e-16,0.0,0.0,GROCERY
481329,0,1.000000e-02,0.0,0.0,DRUG GM
542027,0,1.000000e-02,0.0,0.0,PRODUCE
637766,0,5.820000e+00,0.0,0.0,PRODUCE


In [ ]:
# removing rows where QUANTITY is zero
before = len(df_clean)
df_clean = df_clean[df_clean["QUANTITY"] > 0]
print(before, len(df_clean), before - len(df_clean))   # last number should be 14406

2566555 2552149 14406


### 6. Dropped: zero-quantity rows
- Rows removed: 14,406 (~0.56% of rows)
- Pattern: QUANTITY = 0. Nearly all also have SALES_VALUE = 0; a small
  number have tiny amounts (mostly floating-point noise or cents).
  One produce row had $5.82.
- Reason: Nothing was sold, so these rows carry no information about
  purchases or discounts.
- Note: The fully discounted items (quantity ≥ 1, SALES_VALUE = 0)
  were kept, since quantity was used as the filter.

  SUMMARY : ROWS WENT  FROM 2,595,732 to 2,552,149. 1.7% of data removed

LOOSE ENDS !

In [ ]:
# checking for extreme quantities
(df_clean["QUANTITY"] > 50).sum()

np.int64(16)

In [ ]:

big = df_clean[df_clean["QUANTITY"] > 50].copy()
big["unit_price"] = big["SALES_VALUE"] / big["QUANTITY"]
big[["QUANTITY", "SALES_VALUE", "unit_price", "DEPARTMENT", "COMMODITY_DESC"]].sort_values("QUANTITY", ascending=False)

,QUANTITY,SALES_VALUE,unit_price,DEPARTMENT,COMMODITY_DESC
1541674,144,24.00,0.166667,PRODUCE,CORN
1840749,90,9.00,0.100000,DRUG GM,STATIONERY & SCHOOL SUPPLIES
528062,74,43.01,0.581216,DRUG GM,CANDY - CHECKLANE
621401,72,38.95,0.540972,DRUG GM,CANDY - CHECKLANE
1018143,72,18.00,0.250000,DRUG GM,CANDY - CHECKLANE
2564403,72,8.64,0.120000,DRUG GM,CANDY - PACKAGED
1956516,70,17.50,0.250000,DRUG GM,CANDY - CHECKLANE
2342142,65,6.50,0.100000,DRUG GM,CANDY - PACKAGED
15062,60,82.60,1.376667,GROCERY,ISOTONIC DRINKS
428953,60,12.00,0.200000,GROCERY,PWDR/CRYSTL DRNK MX


### 7. Kept: high-quantity rows (QUANTITY > 50)
- Rows: 16
- Pattern: Bulk purchases of candy, corn, drink mixes and soup.
  Unit prices range from $0.10 to about $2.00, which is plausible
  for each product type.
- Decision: keep since there is no weirdness
- Reason: No sign of data errors, some low unit price can be due to promotions, which are relevant
- Caveat: The two lowest unit prices ($0.10) could be deep
  promotions or pricing errors. They look plausible, so were kept.

In [26]:
(df_clean["DEPARTMENT"].str.strip() == "").sum()

np.int64(0)

### 8. Blank DEPARTMENT values
- Result: 0 rows with a blank DEPARTMENT remain in the cleaned data.
- Explanation: The 15 products with blank departments only appeared
  in rows removed under step 6 (zero quantity).
- No further action needed.

### Summary
- Rows before cleaning: 2,595,732
- Rows after cleaning: 2,552,149
- Total removed: 43,583 (~1.7%)
- Kept on purpose: fully discounted items (QUANTITY ≥ 1, SALES_VALUE = 0)
  and 16 high-quantity rows

In [27]:
df_clean.to_csv("df_clean.csv", index=False)

PHASE 2:

In [1]:
import pandas as pd
import numpy as np

df_clean = pd.read_csv("df_clean.csv")

In [2]:
df_clean.head(5)

,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC,MANUFACTURER,DEPARTMENT,BRAND,COMMODITY_DESC,SUB_COMMODITY_DESC,CURR_SIZE_OF_PRODUCT
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0,69,PRODUCE,Private,POTATOES,POTATOES RUSSET (BULK&BAG),5 LB
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0,2,PRODUCE,National,ONIONS,ONIONS SWEET (BULK&BAG),40 LB
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0,69,PRODUCE,Private,VEGETABLES - ALL OTHERS,CELERY,
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0,2,PRODUCE,National,TROPICAL FRUIT,BANANAS,40 LB
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0,69,PRODUCE,Private,ORGANICS FRUIT & VEGETABLES,ORGANIC CARROTS,1 LB


In [8]:
df_clean.groupby("DEPARTMENT")["SALES_VALUE"].sum().sort_values(ascending=False)

DEPARTMENT
GROCERY            4093810.84
DRUG GM            1055357.16
PRODUCE             557437.89
MEAT                548786.49
MEAT-PCKGD          412436.77
DELI                260865.94
PASTRY              121739.86
NUTRITION            97669.04
SEAFOOD-PCKGD        63093.62
FLORAL               39653.34
COSMETICS            32360.37
SALAD BAR            30953.30
SEAFOOD              29496.92
SPIRITS              21670.88
MISC. TRANS.         10678.69
GARDEN CENTER         7667.82
TRAVEL & LEISUR       2550.38
CHEF SHOPPE           2290.80
RESTAURANT            1986.76
FROZEN GROCERY         653.26
AUTOMOTIVE             452.66
RX                     159.56
GM MERCH EXP           101.10
DAIRY DELI              73.23
CNTRL/STORE SUP         49.05
PHOTO                   41.40
DELI/SNACK BAR          36.08
VIDEO                   34.77
POSTAL CENTER           24.01
VIDEO RENTAL            21.46
PHARMACY SUPPLY         16.92
PORK                    15.70
TOYS                     9.84

In [7]:
df_clean_sales = df_clean.groupby("DEPARTMENT")["SALES_VALUE"].sum().sort_values(ascending=False)

In [9]:
department_sales = df_clean.groupby("DEPARTMENT")["SALES_VALUE"].sum().sort_values(ascending=False)
for department, sales in department_sales.items():
    print(f"{department}: {sales}")

GROCERY: 4093810.84
DRUG GM: 1055357.16
PRODUCE: 557437.89
MEAT: 548786.49
MEAT-PCKGD: 412436.77
DELI: 260865.94
PASTRY: 121739.86
NUTRITION: 97669.04000000001
SEAFOOD-PCKGD: 63093.62
FLORAL: 39653.340000000004
COSMETICS: 32360.37
SALAD BAR: 30953.3
SEAFOOD: 29496.920000000002
SPIRITS: 21670.88
MISC. TRANS.: 10678.69
GARDEN CENTER: 7667.82
TRAVEL & LEISUR: 2550.38
CHEF SHOPPE: 2290.8
RESTAURANT: 1986.76
FROZEN GROCERY: 653.26
AUTOMOTIVE: 452.66
RX: 159.56
GM MERCH EXP: 101.1
DAIRY DELI: 73.23
CNTRL/STORE SUP: 49.05
PHOTO: 41.400000000000006
DELI/SNACK BAR: 36.08
VIDEO: 34.769999999999996
POSTAL CENTER: 24.01
VIDEO RENTAL: 21.46
PHARMACY SUPPLY: 16.92
PORK: 15.7
TOYS: 9.84
HBC: 9.42
CHARITABLE CONT: 7.74
PROD-WHS SALES: 7.52
MEAT-WHSE: 7.0
HOUSEWARES: 2.99
GRO BAKERY: 2.18
ELECT &PLUMBING: 1.0


In [10]:
df_total_sales = df_clean["SALES_VALUE"].sum()
print(f"Total sales value: {df_total_sales}")
share_of_sales = ((df_clean_sales / df_total_sales) * 100).round(1)
print(share_of_sales)


Total sales value: 7392233.76
DEPARTMENT
GROCERY            55.4
DRUG GM            14.3
PRODUCE             7.5
MEAT                7.4
MEAT-PCKGD          5.6
DELI                3.5
PASTRY              1.6
NUTRITION           1.3
SEAFOOD-PCKGD       0.9
FLORAL              0.5
COSMETICS           0.4
SALAD BAR           0.4
SEAFOOD             0.4
SPIRITS             0.3
MISC. TRANS.        0.1
GARDEN CENTER       0.1
TRAVEL & LEISUR     0.0
CHEF SHOPPE         0.0
RESTAURANT          0.0
FROZEN GROCERY      0.0
AUTOMOTIVE          0.0
RX                  0.0
GM MERCH EXP        0.0
DAIRY DELI          0.0
CNTRL/STORE SUP     0.0
PHOTO               0.0
DELI/SNACK BAR      0.0
VIDEO               0.0
POSTAL CENTER       0.0
VIDEO RENTAL        0.0
PHARMACY SUPPLY     0.0
PORK                0.0
TOYS                0.0
HBC                 0.0
CHARITABLE CONT     0.0
PROD-WHS SALES      0.0
MEAT-WHSE           0.0
HOUSEWARES          0.0
GRO BAKERY          0.0
ELECT &PLUMBING     0.0

In [14]:
shares = (df_clean_sales / df_total_sales) * 100
shares.cumsum().round(1)

DEPARTMENT
GROCERY             55.4
DRUG GM             69.7
PRODUCE             77.2
MEAT                84.6
MEAT-PCKGD          90.2
DELI                93.7
PASTRY              95.4
NUTRITION           96.7
SEAFOOD-PCKGD       97.6
FLORAL              98.1
COSMETICS           98.5
SALAD BAR           98.9
SEAFOOD             99.3
SPIRITS             99.6
MISC. TRANS.        99.8
GARDEN CENTER       99.9
TRAVEL & LEISUR     99.9
CHEF SHOPPE         99.9
RESTAURANT         100.0
FROZEN GROCERY     100.0
AUTOMOTIVE         100.0
RX                 100.0
GM MERCH EXP       100.0
DAIRY DELI         100.0
CNTRL/STORE SUP    100.0
PHOTO              100.0
DELI/SNACK BAR     100.0
VIDEO              100.0
POSTAL CENTER      100.0
VIDEO RENTAL       100.0
PHARMACY SUPPLY    100.0
PORK               100.0
TOYS               100.0
HBC                100.0
CHARITABLE CONT    100.0
PROD-WHS SALES     100.0
MEAT-WHSE          100.0
HOUSEWARES         100.0
GRO BAKERY         100.0
ELECT &PLUMBIN

In [ ]:
# investigate the MISC. TRANS. department
m = df_clean[df_clean["DEPARTMENT"] == "MISC. TRANS."]
print(m["COMMODITY_DESC"].value_counts().head(10))
print(len(m), m["SALES_VALUE"].sum())

COMMODITY_DESC
NO COMMODITY DESCRIPTION    2292
Name: count, dtype: int64
2292 10678.69


In [16]:
m = df_clean[df_clean["DEPARTMENT"] == "MISC. TRANS."]
print(m["SUB_COMMODITY_DESC"].value_counts().head(10))
print(m["PRODUCT_ID"].nunique())
print(m[["QUANTITY", "SALES_VALUE", "RETAIL_DISC"]].describe())

SUB_COMMODITY_DESC
NO SUBCOMMODITY DESCRIPTION       2159
STRAWBERRIES                        27
BEERALEMALT LIQUORS                 23
SFT DRNK MLT-PK BTL CARB (EXCP      18
APPLES RED DELICIOUS (BULK&BAG       7
DISTILLED WATER                      4
PINEAPPLE WHOLE&PEEL/CORED           4
ORGANIC APPLES                       4
DOLLAR DAYS                          3
DISPOSABLE FOILWARE                  3
Name: count, dtype: int64
467
          QUANTITY  SALES_VALUE  RETAIL_DISC
count  2292.000000  2292.000000  2292.000000
mean      1.535777     4.659114    -0.140838
std       1.646136    24.496516     1.632551
min       1.000000     0.000000   -70.000000
25%       1.000000     0.500000     0.000000
50%       1.000000     1.570000     0.000000
75%       1.000000     2.990000     0.000000
max      30.000000   505.000000     0.000000


In [ ]:
# investigate outliers in the MISC. TRANS. department
m.sort_values("SALES_VALUE", ascending=False).head(5)[["QUANTITY", "SALES_VALUE", "RETAIL_DISC", "SUB_COMMODITY_DESC"]]

,QUANTITY,SALES_VALUE,RETAIL_DISC,SUB_COMMODITY_DESC
645638,2,505.0,0.0,NO SUBCOMMODITY DESCRIPTION
539038,1,455.0,0.0,NO SUBCOMMODITY DESCRIPTION
518242,1,430.0,0.0,NO SUBCOMMODITY DESCRIPTION
3579,1,400.0,0.0,NO SUBCOMMODITY DESCRIPTION
2312959,1,380.0,0.0,NO SUBCOMMODITY DESCRIPTION


In [18]:
big = df_clean[df_clean["SALES_VALUE"] > 100]
print(len(big), big["DEPARTMENT"].value_counts().head(5))

99 DEPARTMENT
DRUG GM         50
MISC. TRANS.    15
MEAT            13
GROCERY          9
FLORAL           3
Name: count, dtype: int64


### 9. [Kept / Excluded]: MISC. TRANS. department
- Rows: 2,292 (~$10.7K, ~0.14% of sales)
- Pattern: ...
- Decision: ...
- Reason: ...
- Caveat: ...